In [8]:
SKIP_LLM_CHUNKING = True

## Ingestion


In [1]:
import io
import zipfile
import requests
import frontmatter

def read_repo_data(repo_owner, repo_name):
    """
    Download and parse all markdown files from a GitHub repository.
    
    Args:
        repo_owner: GitHub username or organization
        repo_name: Repository name
    
    Returns:
        List of dictionaries containing file content and metadata
    """
    prefix = 'https://codeload.github.com' 
    url = f'{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/main'
    resp = requests.get(url)
    
    if resp.status_code != 200:
        raise Exception(f"Failed to download repository: {resp.status_code}")

    repository_data = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))
    
    for file_info in zf.infolist():
        filename = file_info.filename
        filename_lower = filename.lower()

        if not (filename_lower.endswith('.md') 
            or filename_lower.endswith('.mdx')):
            continue
    
        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode('utf-8', errors='ignore')
                post = frontmatter.loads(content)
                data = post.to_dict()
                data['filename'] = filename
                repository_data.append(data)
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue
    
    zf.close()
    return repository_data

In [2]:
dtc_faq = read_repo_data('DataTalksClub', 'faq')
evidently_docs = read_repo_data('evidentlyai', 'docs')

print(f"FAQ documents: {len(dtc_faq)}")
print(f"Evidently documents: {len(evidently_docs)}")

FAQ documents: 1285
Evidently documents: 95


## Chunking and Preprocessing

### Intelligent Chunking with LLM

In [10]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

# now you can access them
openai_key = os.getenv("OPENAI_API_KEY")
print("Has key?", bool(openai_key))

if not SKIP_LLM_CHUNKING:
    openai_client = OpenAI(openai_key)

    def llm(prompt, model='gpt-4o-mini'):
        messages = [
            {"role": "user", "content": prompt}
        ]

        response = openai_client.responses.create(
            model='gpt-4o-mini',
            input=messages
        )

        return response.output_text


Has key? False


In [ ]:
prompt_template = """
Split the provided document into logical sections
that make sense for a Q&A system.

Each section should be self-contained and cover
a specific topic or concept.

<DOCUMENT>
{document}
</DOCUMENT>

Use this format:

## Section Name

Section content with all relevant details

---

## Another Section Name

Another section content

---
""".strip()


In [ ]:
if not SKIP_LLM_CHUNKING:
    def intelligent_chunking(text):
        prompt = prompt_template.format(document=text)
        response = llm(prompt)
        sections = response.split('---')
        sections = [s.strip() for s in sections if s.strip()]
        return sections

In [12]:
from tqdm.auto import tqdm

if not SKIP_LLM_CHUNKING:
    evidently_chunks = []

    for doc in tqdm(evidently_docs):
        doc_copy = doc.copy()
        doc_content = doc_copy.pop('content')

        sections = intelligent_chunking(doc_content)
        for section in sections:
            section_doc = doc_copy.copy()
            section_doc['section'] = section
            evidently_chunks.append(section_doc)

### Simple Chunking

In [3]:
def sliding_window(seq, size, step):
  if size <= 0 or step <= 0:
    raise ValueError("size and step must be positive")

  n = len(seq)
  result = []

  for i in range(0, n, step):
    chunk = seq[i:i+size]
    result.append({'start': i, 'chunk': chunk})

    if i + size >= n:
      break
  
  return result

In [5]:
from tqdm.auto import tqdm

evidently_chunks = []
for doc in evidently_docs:
  doc_copy = doc.copy()
  doc_content = doc_copy.pop('content')
  chunks = sliding_window(doc_content, 2000, 1000)

  for chunk in chunks:
    chunk.update(doc_copy)

  evidently_chunks.extend(chunks)
  
print(f"Evidently chunks: {len(evidently_chunks)}")

Evidently chunks: 576


## Search

### Text search

In [21]:
from minsearch import Index

evidently_index = Index(
    text_fields=["chunk", "title", "description", "filename"],
    keyword_fields=[]
)

evidently_index.fit(evidently_chunks)

In [ ]:
query = 'What should be in a test dataset for AI evaluation?'
results = evidently_index.search(query)

print(f"Top result: {results[0]['chunk'][:500]}...")

Top result: Retrieval-Augmented Generation (RAG) systems rely on retrieving answers from a knowledge base before generating responses. To evaluate them effectively, you need a test dataset that reflects what the system *should* know.

Instead of manually creating test cases, you can generate them directly from your knowledge source, ensuring accurate and relevant ground truth data.

## Create a RAG test dataset

You can generate ground truth RAG dataset from your data source.

### 1. Create a Project

In th...


### Vector search

In [15]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('multi-qa-distilbert-cos-v1')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/523 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/333 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [20]:
from minsearch import VectorSearch
from tqdm.auto import tqdm
import numpy as np

evidently_embeddings = []

for d in tqdm(evidently_chunks):
  v = embedding_model.encode(d['chunk'])
  evidently_embeddings.append(v)

evidently_embeddings = np.array(evidently_embeddings)

evidently_vindex = VectorSearch()
evidently_vindex.fit(evidently_embeddings, evidently_chunks)

  0%|          | 0/576 [00:00<?, ?it/s]

In [22]:
print(evidently_embeddings.shape)
print(len(evidently_chunks))

(576, 768)
576


In [ ]:
query = 'What should be in a test dataset for AI evaluation?'
q = embedding_model.encode(query)
results = evidently_vindex.search(q)

print(f"Top result: {results[0]['chunk'][:500]}...")

### Hybrid search

In [ ]:
def text_search(query):
  return evidently_index.search(query, num_results=5)

def vector_search(query):
  q = embedding_model.encode(query)
  return evidently_vindex.search(q, num_results=5)

def hybrid_search(query):
  text_results = text_search(query)
  vector_results = vector_search(query)

  # Combine and deduplicate results
  seen_ids = set()
  combined_results = []

  for result in text_results + vector_results:
    if result['filename'] not in seen_ids:
      seen_ids.add(result['filename'])
      combined_results.append(result)

  return combined_results